# 🚗 YOLO11s (Small) Yüksek Çözünürlüklü Araç Tespiti Eğitimi (1024px)
### Bitirme Projesi - Model Kapasitesi ve Küçük Araç Tespiti İyileştirmesi

Bu notebook, araç tespit doğruluğunu ve küçük/uzak araçların yakalanma başarısını artırmak için **YOLO11s (Small - 9.4M Parametre)** modelini **1024x1024 çözünürlük** ve optimize edilmiş transfer learning stratejisiyle eğitmek üzere hazırlanmıştır.

**Bu Sürümdeki Kritik İyileştirmeler:**
1. **`yolo11s.pt` (Small Model - 9.4M Parametre):** Nano modele göre ~3.6 kat daha fazla parametre kapasitesi sayesinde karmaşık araç silüetlerini ve örtüşen (occluded) araçları çok daha yüksek kesinlikle tanır.
2. **`imgsz=1024` (Yüksek Çözünürlük):** Trafik kameralarındaki uzaktaki küçük arabaların piksel kaybını önler, tespit oranını önemli ölçüde artırır.
3. **`freeze=10` (Omurga Dondurma):** Modelin COCO 118K veri setinden öğrendiği zengin görsel filtreler dondurulur, katastrofik unutma engellenir.
4. **`lr0=0.0005` (Hassas İnce Ayar):** Ağırlıkların sarsılmaması için öğrenme hızı nazik bir seviyeye ayarlanmıştır.
5. **`batch=8`:** 1024px yüksek çözünürlükte GPU belleğini (VRAM) güvenli ve verimli kullanır.
6. **Otomatik Paketleme:** Eğitim tamamlandığında sonuçlar ve `best.pt` ağırlığı doğrudan `yolo11_results.zip` adıyla bilgisayarınıza indirilir.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aydin-eng/federated-traffic-learning/blob/main/traffic_vehicle_detection_yolo11.ipynb)


## 1. Donanım ve GPU Kontrolü
Google Colab üzerinde GPU hızlandırıcısının (T4 / V100 / A100) devrede olduğunu kontrol ediyoruz.


In [13]:
!nvidia-smi


Thu Nov  5 09:19:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             35W /   70W |     993MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Deponun Klonlanması ve Çalışma Ortamı Kurulumu
GitHub üzerindeki güncel veri setimizi ve yapılandırma dosyalarımızı Colab ortamına çekiyoruz.


In [14]:
import os

# Eğer Colab üzerinde çalışıyorsak repoyu klonla
if not os.path.exists('federated-traffic-learning'):
    !git clone https://github.com/aydin-eng/federated-traffic-learning.git
    %cd federated-traffic-learning
else:
    print('Repo zaten mevcut, güncelleniyor...')
    %cd federated-traffic-learning
    !git pull

print('Mevcut çalışma dizini:', os.getcwd())


Cloning into 'federated-traffic-learning'...
remote: Enumerating objects: 803, done.
remote: Counting objects: 100% (286/286), done.
remote: Compressing objects: 100% (262/262), done.
remote: Total 803 (delta 16), reused 280 (delta 11), pack-reused 517 (from 1)
Receiving objects: 100% (803/803), 130.89 MiB | 8.17 MiB/s, done.
Resolving deltas: 100% (54/54), done.
/content/federated-traffic-learning/federated-traffic-learning
Mevcut çalışma dizini: /content/federated-traffic-learning/federated-traffic-learning


## 3. Gerekli Kütüphanelerin Yüklenmesi
Ultralytics (YOLO11) kütüphanesini yüklüyoruz.


In [15]:
!pip install -q ultralytics


## 4. Veri Seti ve YAML Konfigürasyonu Doğrulama
`vehicle_dataset_v3_augmented` veri setimizin ve `data.yaml` dosyamızın yollarını kontrol ediyoruz.


In [16]:
import yaml

yaml_path = 'vehicle_dataset_v3_augmented/data.yaml'
with open(yaml_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

print('Veri Konfigürasyonu:')
for k, v in cfg.items():
    print(f' - {k}: {v}')


Veri Konfigürasyonu:
 - path: vehicle_dataset_v3_augmented
 - train: images/train
 - val: images/val
 - test: images/test
 - names: {0: 'car', 1: 'motorcycle', 2: 'bus', 3: 'truck'}


## 5. YOLO11s (Small) Modelinin Başlatılması
Ultralytics üzerinden önceden eğitilmiş yüksek kapasiteli **YOLO11s (Small - 9.4M Parametre)** modelini yüklüyoruz.


In [17]:
from ultralytics import YOLO

# Yüksek kapasiteli YOLO11s (Small) modelini yükle
model = YOLO('yolo11s.pt')
print('Base YOLO11s (Small) modeli başarıyla yüklendi!')
model.info()


Base YOLO11s (Small) modeli başarıyla yüklendi!
YOLO11s summary: 181 layers, 9,458,752 parameters, 0 gradients, 21.8 GFLOPs


(181, 9458752, 0, 21.8363392)

## 6. Yüksek Çözünürlüklü Model Eğitimi (1024px Fine-Tuning) 🚀
Küçük araçları yakalamak ve tespit doğruluğunu maksimize etmek için optimize parametrelerle eğitimi başlatıyoruz:

* **`model='yolo11s.pt'`**: 9.4M parametreli güçlü model.
* **`imgsz=1024`**: Yüksek çözünürlük sayesinde uzaktaki araçlar kaybolmaz.
* **`freeze=10`**: İlk 10 omurga katmanı dondurulur (COCO görsel filtreleri korunur).
* **`lr0=0.0005`**: Hassas ve sarsıntısız öğrenme oranı.
* **`batch=8`**: 1024px çözünürlük için ideal GPU bellek kullanımı.
* **`cos_lr=True` & `warmup_epochs=3`**: Dengeli öğrenme eğrisi.
* **`epochs=60` & `patience=20`**: Early stopping ile en iyi mAP noktasında durdurma.


In [18]:
# Yüksek çözünürlüklü eğitimi başlat
results = model.train(
    data='vehicle_dataset_v3_augmented/data.yaml',
    epochs=60,
    imgsz=1024,                 # 1024px yüksek çözünürlük
    batch=8,                    # GPU VRAM koruması
    device=0,                   # Colab GPU
    project='runs/train',
    name='yolo11s_vehicle_exp',
    freeze=10,                  # Omurga katmanlarını dondur
    lr0=0.0005,                 # Hassas ince ayar öğrenme hızı
    lrf=0.01,
    cos_lr=True,                # Yumuşak öğrenme çizelgesi
    warmup_epochs=3,            # Isınma turları
    mosaic=0.5,                 # Dengeli veri zenginleştirme
    close_mosaic=10,            # Son 10 epoch mozaik kapatma
    patience=20,                # Erken durdurma toleransı
    save=True,
    plots=True,
    seed=42,
    verbose=True
)


Ultralytics 8.4.202 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=vehicle_dataset_v3_augmented/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=yolo11s_vehicle_exp, nbs=64, nms=None, opset=None, 

## 7. Model Başarısının Test Kümesinde Değerlendirilmesi (Evaluation)
Eğitilen en iyi ağırlık (`best.pt`) ile test kümesi (`images/test`) üzerindeki mAP50, mAP50-95, Precision ve Recall değerlerini ölçüyoruz.


In [19]:
# Test kümesi üzerinde değerlendirme
metrics = model.val(split='test', imgsz=1024)

print('\n' + '=' * 45)
print('  TEST KÜMESİ DEĞERLENDİRME SONUÇLARI (YOLO11s - 1024px)')
print('=' * 45)
print(f"mAP@50:     {metrics.box.map50:.4f}")
print(f"mAP@50-95:  {metrics.box.map:.4f}")
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")
print('=' * 45)


Ultralytics 8.4.202 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 100 layers, 9,414,348 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1259.3±238.5 MB/s, size: 33.9 KB)
val: Scanning /content/federated-traffic-learning/federated-traffic-learning/vehicle_dataset_v3_augmented/labels/test... 19 images, 16 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 19/19 2.8Kit/s 0.0s
val: New cache created: /content/federated-traffic-learning/federated-traffic-learning/vehicle_dataset_v3_augmented/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.6it/s 1.2s
                   all         19          5      0.438          1      0.787      0.364
                   car          2          2      0.271          1      0.828      0.663
            motorcycle          1          1      0.848          1      0.995      0.298
                   bus   

## 8. Eğitim Metrikleri ve Karışıklık Matrisi Grafikleri
Kayıp eğrilerini, PR eğrisini ve sınıflara göre Confusion Matrix grafiğini görüntülüyoruz.


In [20]:
import glob
from IPython.display import Image, display

plots = [
    'runs/train/yolo11s_vehicle_exp/results.png',
    'runs/train/yolo11s_vehicle_exp/confusion_matrix_normalized.png',
    'runs/train/yolo11s_vehicle_exp/BoxPR_curve.png'
]

for p in plots:
    if os.path.exists(p):
        print(f'Görsel: {os.path.basename(p)}')
        display(Image(filename=p, width=700))


## 9. Test ve Özel Trafik Fotoğrafı Üzerinde Model Çıkarımı (Inference)
Modelin hem test kümesindeki hem de yüklediğimiz özel test fotoğrafındaki (`test_samples/user_test_photo.jpg`) tespitlerini çizdiriyoruz.


In [21]:
# 1. Test kümesindeki görseller üzerinde tahmin yap
test_preds = model.predict(
    source='vehicle_dataset_v3_augmented/images/test',
    conf=0.25,
    imgsz=1024,
    save=True,
    project='runs/detect',
    name='test_results'
)

# 2. Özel test fotoğrafı varsa onu da tahmin et
custom_photo = 'test_samples/user_test_photo.jpg'
if os.path.exists(custom_photo):
    custom_pred = model.predict(
        source=custom_photo,
        conf=0.25,
        imgsz=1024,
        save=True,
        project='runs/detect',
        name='custom_test_result'
    )
    pred_custom_imgs = glob.glob('runs/detect/custom_test_result/*.jpg')
    if pred_custom_imgs:
        print('Özel test fotoğrafı sonucu:')
        display(Image(filename=pred_custom_imgs[0], width=700))

# İlk 2 test kümesi sonucunu görselleştir
pred_images = glob.glob('runs/detect/test_results/*.jpg')
print(f'Toplam {len(pred_images)} test görseli işlendi. Örnek sonuçlar:')
for img_path in pred_images[:2]:
    display(Image(filename=img_path, width=650))



image 1/19 /content/federated-traffic-learning/federated-traffic-learning/vehicle_dataset_v3_augmented/images/test/000000000074.jpg: 704x1024 2 cars, 39.4ms
image 2/19 /content/federated-traffic-learning/federated-traffic-learning/vehicle_dataset_v3_augmented/images/test/000000000078.jpg: 1024x1024 (no detections), 51.8ms
image 3/19 /content/federated-traffic-learning/federated-traffic-learning/vehicle_dataset_v3_augmented/images/test/000000000089.jpg: 768x1024 (no detections), 51.2ms
image 4/19 /content/federated-traffic-learning/federated-traffic-learning/vehicle_dataset_v3_augmented/images/test/000000000094.jpg: 704x1024 3 cars, 23.1ms
image 5/19 /content/federated-traffic-learning/federated-traffic-learning/vehicle_dataset_v3_augmented/images/test/000000000143.jpg: 864x1024 (no detections), 46.9ms
image 6/19 /content/federated-traffic-learning/federated-traffic-learning/vehicle_dataset_v3_augmented/images/test/000000000260.jpg: 704x1024 4 cars, 1 bus, 2 trucks, 23.7ms
image 7/19 /

## 10. Sonuçların Otomatik Paketlenmesi ve İndirilmesi 📦
Eğitilen `best.pt` ağırlık dosyasını ve tüm grafikleri tek bir `yolo11_results.zip` dosyası haline getirip bilgisayarınıza indirir.


In [23]:
import shutil
from google.colab import files

best_weights = '/content/federated-traffic-learning/runs/detect/runs/train/yolo11n_vehicle_exp/weights/best.pt'
if os.path.exists(best_weights):
    size_mb = os.path.getsize(best_weights) / (1024 * 1024)
    print(f'✅ En başarılı model ağırlığı hazır: {best_weights} ({size_mb:.2f} MB)')

print('📦 Sonuçlar paketleniyor ve yolo11_results.zip indiriliyor...')
shutil.make_archive('yolo11_results', 'zip', '/content/federated-traffic-learning/runs/detect/runs/train/yolo11n_vehicle_exp')
files.download('yolo11_results.zip')
print('🚀 yolo11_results.zip indirme başlatıldı! İndirilen dosyayı masaüstündeki proje klasörünüze atabilirsiniz.')


✅ En başarılı model ağırlığı hazır: /content/federated-traffic-learning/runs/detect/runs/train/yolo11n_vehicle_exp/weights/best.pt (5.25 MB)
📦 Sonuçlar paketleniyor ve yolo11_results.zip indiriliyor...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🚀 yolo11_results.zip indirme başlatıldı! İndirilen dosyayı masaüstündeki proje klasörünüze atabilirsiniz.
